# Framingham Heart Study: 10-Year CHD Risk Prediction


## 1. Introduction

This Jupyter notebook presents a comprehensive machine learning project aimed at predicting the 10-year risk of Coronary Heart Disease (CHD) based on the Framingham Heart Study dataset. Coronary Heart Disease is a significant health concern globally, and early prediction can enable timely interventions and lifestyle modifications.

The objective of this project is to:
*   Perform thorough Exploratory Data Analysis (EDA) to understand the dataset's characteristics.
*   Preprocess the data, handling missing values and outliers.
*   Engineer relevant features to enhance model performance.
*   Build and evaluate classification models to predict the `TenYearCHD` target variable.
*   Identify the best-performing model and save it for future use.
*   Provide insights into the factors contributing to CHD risk.

The `TenYearCHD` variable is a binary indicator (0 = No CHD, 1 = CHD), making this a binary classification problem.


## 2. Architecture Diagram

To illustrate the overall machine learning pipeline, an architecture diagram is essential. This diagram outlines the flow of data from ingestion to model deployment.

**(Note: Please create this diagram using https://app.diagrams.net/ and download it as an SVG file. The diagram should visually represent the following stages:)**

*   **Data Source:** Raw Dataset (e.g., `framingham.csv`)
*   **Data Loading:** Jupyter Notebook / Python Script
*   **Data Preprocessing:**
    *   Missing Value Imputation
    *   Outlier Handling
    *   Feature Engineering
    *   Feature Scaling
*   **EDA:** Visualization, Summary Statistics
*   **Feature Selection:** Based on EDA, Correlation
*   **Data Split:** Training, Validation, Test Sets
*   **Model Training:**
    *   Classification Models (e.g., Logistic Regression, RandomForest, Gradient Boosting)
    *   Hyperparameter Tuning
*   **Model Evaluation:**
    *   Metrics (Accuracy, Precision, Recall, F1-score, ROC-AUC)
    *   Visualizations (Confusion Matrix, ROC Curve)
*   **Model Selection:** Best Performing Model
*   **Model Saving:** Serialized Model (e.g., `model.pkl` in `artifacts` directory)
*   **Prediction:** New Data Input -> Loaded Model -> Prediction Output
*   **Logging:** All stages write to `ml_logs` directory.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import zscore

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline # Use Imblearn's Pipeline to handle SMOTE before cross-validation

import warnings
import logging
import os
import pickle

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# --- Logging Setup ---
# Create ml_logs directory if it doesn't exist
log_dir = 'ml_logs'
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

# Configure logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[
                        logging.FileHandler(os.path.join(log_dir, 'ml_pipeline.log')),
                        logging.StreamHandler() # Also log to console
                    ])

logging.info("Starting the ML pipeline notebook execution.")

# --- Error Handling Helper ---
def handle_exception(e, stage):
    logging.error(f"An error occurred during {stage}: {e}", exc_info=True)
    print(f"ERROR: An error occurred during {stage}. Check logs for details.")

# Define the data path
DATA_PATH = 'data/framingham.csv'
ARTIFACTS_PATH = 'artifacts'

# Create artifacts directory if it doesn't exist
if not os.path.exists(ARTIFACTS_PATH):
    os.makedirs(ARTIFACTS_PATH)
    logging.info(f"Created directory: {ARTIFACTS_PATH}")


2025-10-15 00:01:04,902 - INFO - Starting the ML pipeline notebook execution.
2025-10-15 00:01:04,918 - INFO - Created directory: artifacts


## 3. Data Loading

In this section, we load the dataset into a pandas DataFrame. The dataset is expected to be in a CSV format within the `data` directory. We'll also perform an initial check of its structure.


In [ ]:
try:
    # Ensure the data directory exists and the file is there (for demonstration)
    if not os.path.exists('data'):
        os.makedirs('data')
        logging.warning("Created 'data' directory. Please place 'framingham.csv' inside it.")

    # Create a dummy CSV file for demonstration if it doesn't exist
    if not os.path.exists(DATA_PATH):
        sample_data = [
            {'male': 1, 'age': 39, 'education': 4.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 195.0, 'sysBP': 106.0, 'diaBP': 70.0, 'BMI': 26.97, 'heartRate': 80.0, 'glucose': 77.0, 'TenYearCHD': 0},
            {'male': 0, 'age': 46, 'education': 2.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 250.0, 'sysBP': 121.0, 'diaBP': 81.0, 'BMI': 28.73, 'heartRate': 95.0, 'glucose': 76.0, 'TenYearCHD': 0},
            {'male': 1, 'age': 48, 'education': 1.0, 'currentSmoker': 1, 'cigsPerDay': 20.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 245.0, 'sysBP': 127.5, 'diaBP': 80.0, 'BMI': 25.34, 'heartRate': 75.0, 'glucose': 70.0, 'TenYearCHD': 0},
            {'male': 0, 'age': 61, 'education': 3.0, 'currentSmoker': 1, 'cigsPerDay': 30.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 225.0, 'sysBP': 150.0, 'diaBP': 95.0, 'BMI': 28.58, 'heartRate': 65.0, 'glucose': 103.0, 'TenYearCHD': 1},
            {'male': 0, 'age': 46, 'education': 3.0, 'currentSmoker': 1, 'cigsPerDay': 23.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 285.0, 'sysBP': 130.0, 'diaBP': 84.0, 'BMI': 23.1, 'heartRate': 85.0, 'glucose': 85.0, 'TenYearCHD': 0},
            {'male': 0, 'age': 43, 'education': 2.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 228.0, 'sysBP': 180.0, 'diaBP': 110.0, 'BMI': 30.3, 'heartRate': 77.0, 'glucose': 99.0, 'TenYearCHD': 0},
            {'male': 0, 'age': 63, 'education': 1.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 205.0, 'sysBP': 138.0, 'diaBP': 71.0, 'BMI': 33.11, 'heartRate': 60.0, 'glucose': 85.0, 'TenYearCHD': 1},
            {'male': 0, 'age': 45, 'education': 2.0, 'currentSmoker': 1, 'cigsPerDay': 20.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 313.0, 'sysBP': 100.0, 'diaBP': 71.0, 'BMI': 21.68, 'heartRate': 79.0, 'glucose': 78.0, 'TenYearCHD': 0},
            {'male': 1, 'age': 52, 'education': 1.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 260.0, 'sysBP': 141.5, 'diaBP': 89.0, 'BMI': 26.36, 'heartRate': 76.0, 'glucose': 79.0, 'TenYearCHD': 0},
            {'male': 1, 'age': 43, 'education': 1.0, 'currentSmoker': 1, 'cigsPerDay': 30.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 225.0, 'sysBP': 162.0, 'diaBP': 107.0, 'BMI': 23.61, 'heartRate': 93.0, 'glucose': 88.0, 'TenYearCHD': 0}
        ]
        # Add more rows to make it a "full" dataset if possible, or just repeat for a larger sample
        # For a truly full dataset, you'd usually have a separate, larger file.
        # For this demonstration, I'll repeat the sample data to simulate a larger dataset.
        full_sample_data = sample_data * 200 # Simulate ~2000 rows
        df_sample = pd.DataFrame(full_sample_data)
        df_sample.to_csv(DATA_PATH, index=False)
        logging.info(f"Created a sample '{os.path.basename(DATA_PATH)}' file for demonstration.")

    df = pd.read_csv(DATA_PATH)
    logging.info(f"Dataset loaded successfully from {DATA_PATH}. Shape: {df.shape}")
    print(f"Dataset loaded successfully. Shape: {df.shape}")
    print("\nFirst 5 rows of the dataset:")
    print(df.head())

except FileNotFoundError:
    handle_exception(f"The file '{DATA_PATH}' was not found. Please ensure it's in the 'data' directory.", "Data Loading")
except pd.errors.EmptyDataError:
    handle_exception(f"The file '{DATA_PATH}' is empty.", "Data Loading")
except pd.errors.ParserError:
    handle_exception(f"Error parsing the CSV file '{DATA_PATH}'. Check file format.", "Data Loading")
except Exception as e:
    handle_exception(e, "Data Loading")


## 4. Exploratory Data Analysis (EDA)

EDA is a crucial step to understand the dataset's characteristics, identify patterns, anomalies, and relationships between features.


In [ ]:
try:
    logging.info("Starting EDA: Initial data overview.")
    print("\nDataset Information:")
    df.info()

    print("\nDescriptive Statistics for Numerical Features:")
    print(df.describe())

    print("\nMissing Values Count:")
    print(df.isnull().sum())
    missing_percentage = df.isnull().sum() * 100 / len(df)
    print("\nMissing Values Percentage:")
    print(missing_percentage)

    print("\nNumber of duplicate rows:", df.duplicated().sum())

    # Check distribution of target variable
    print("\nDistribution of 'TenYearCHD':")
    print(df['TenYearCHD'].value_counts())
    print(f"Percentage of CHD cases: {df['TenYearCHD'].value_counts(normalize=True)[1]:.2%}")
    logging.info(f"Target variable distribution: {df['TenYearCHD'].value_counts().to_dict()}")

    # Identify categorical and numerical columns based on schema and observed values
    numerical_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
    # 'education', 'BPMeds' are float but are categorical/binary and will be handled during preprocessing.
    # 'male', 'currentSmoker', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'TenYearCHD' are binary/categorical.

    # Check unique values for columns that might be categorical but are int/float
    for col in ['male', 'education', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'TenYearCHD']:
        if col in df.columns:
            print(f"\nUnique values for '{col}': {df[col].unique()}")

    logging.info("EDA: Initial data overview completed.")

except Exception as e:
    handle_exception(e, "EDA Initial Overview")


## 5. Preprocessing

This section covers data cleaning and transformation steps, including handling missing values, outliers, and feature engineering.


In [ ]:
try:
    logging.info("Starting Data Preprocessing.")

    # 1. Handle Missing Values
    # Impute numerical features with the median.
    # For 'cigsPerDay', 'BPMeds', 'totChol', 'BMI', 'heartRate', 'glucose', 'education'
    # 'education' is ordinal, median imputation is reasonable. 'BPMeds' is binary, median is 0.0 or 1.0, so mode might be better.
    # Let's use median for most numerical and mode for binary/ordinal if appropriate.

    impute_median_cols = ['totChol', 'BMI', 'heartRate', 'glucose', 'cigsPerDay']
    impute_mode_cols = ['education', 'BPMeds'] # These are float but represent categories/binary

    for col in impute_median_cols:
        if df[col].isnull().any():
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            logging.info(f"Missing values in '{col}' imputed with median: {median_val}")

    for col in impute_mode_cols:
        if df[col].isnull().any():
            mode_val = df[col].mode()[0] # .mode() can return multiple values if multimodal
            df[col].fillna(mode_val, inplace=True)
            logging.info(f"Missing values in '{col}' imputed with mode: {mode_val}")

    print("\nMissing values after imputation:")
    print(df.isnull().sum())

    # 2. Correct Data Types
    # Convert binary/categorical features to 'category' or 'int'
    binary_categorical_cols = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'TenYearCHD']
    for col in binary_categorical_cols:
        if col in df.columns:
            df[col] = df[col].astype(int) # Ensure they are integers 0/1
    if 'education' in df.columns:
        df['education'] = df['education'].astype(int) # Convert education to int as it's ordinal

    logging.info("Data types corrected for binary/categorical features.")
    print("\nData types after correction:")
    print(df.dtypes)

    # 3. Feature Engineering
    # Create Age Groups
    df['age_group'] = pd.cut(df['age'], bins=[20, 30, 40, 50, 60, 70, 80, np.inf],
                             labels=['20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80+'], right=False)
    logging.info("Feature 'age_group' created.")

    # Create BMI Categories
    def bmi_category(bmi):
        if bmi < 18.5: return 'Underweight'
        elif 18.5 <= bmi < 24.9: return 'Normal'
        elif 24.9 <= bmi < 29.9: return 'Overweight'
        else: return 'Obese'
    df['bmi_category'] = df['BMI'].apply(bmi_category)
    logging.info("Feature 'bmi_category' created.")

    # Create Blood Pressure Status (simplified)
    # Systolic BP: Normal (<120), Elevated (120-129), High Stage 1 (130-139), High Stage 2 (>=140)
    # Diastolic BP: Normal (<80), Elevated (80-89), High Stage 2 (>=90)
    def bp_status(sysBP, diaBP):
        if sysBP < 120 and diaBP < 80: return 'Normal BP'
        elif (120 <= sysBP < 130 and diaBP < 80) or (sysBP < 120 and 80 <= diaBP < 90): return 'Elevated BP'
        elif (130 <= sysBP < 140 or 80 <= diaBP < 90): return 'High BP Stage 1'
        elif (sysBP >= 140 or diaBP >= 90): return 'High BP Stage 2'
        else: return 'Unknown BP' # Should not happen with valid data
    df['bp_status'] = df.apply(lambda row: bp_status(row['sysBP'], row['diaBP']), axis=1)
    logging.info("Feature 'bp_status' created.")

    # Convert new categorical features to one-hot encoding
    df = pd.get_dummies(df, columns=['age_group', 'bmi_category', 'bp_status'], drop_first=True)
    logging.info("New categorical features one-hot encoded.")

    # 4. Outlier Handling (for numerical features)
    # Using IQR method for outlier detection and capping
    numerical_features_for_outlier = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
    for col in numerical_features_for_outlier:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers_count = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
        if outliers_count > 0:
            df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
            df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
            logging.info(f"Outliers in '{col}' handled by capping. Original outliers count: {outliers_count}")
        else:
            logging.info(f"No significant outliers found in '{col}' based on IQR method.")

    logging.info("Data Preprocessing completed.")
    print("\nDataset head after preprocessing and feature engineering:")
    print(df.head())

except Exception as e:
    handle_exception(e, "Data Preprocessing")


## 6. Visual Representation of EDA (Plotly)

Visualizations help to deeply understand data distributions, relationships, and potential issues. Plotly is used for interactive and informative plots.


In [ ]:
try:
    logging.info("Starting Visual Representation of EDA.")

    # Distribution of target variable
    fig = px.pie(df, names='TenYearCHD', title='Distribution of TenYearCHD (Target Variable)', hole=0.3)
    fig.update_traces(textinfo='percent+label', marker=dict(colors=['lightblue', 'orange']))
    fig.show()
    logging.info("Plotly: Distribution of TenYearCHD displayed.")

    # Distributions of numerical features
    numerical_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
    fig = make_subplots(rows=len(numerical_cols)//2 + len(numerical_cols)%2, cols=2,
                        subplot_titles=[f'Distribution of {col}' for col in numerical_cols])
    row_num = 1
    col_num = 1
    for col in numerical_cols:
        fig.add_trace(go.Histogram(x=df[col], name=col, marker_color='cadetblue'), row=row_num, col=col_num)
        fig.update_xaxes(title_text=col, row=row_num, col=col_num)
        fig.update_yaxes(title_text="Count", row=row_num, col=col_num)
        col_num += 1
        if col_num > 2:
            col_num = 1
            row_num += 1
    fig.update_layout(height=400 * row_num, showlegend=False, title_text="Distributions of Numerical Features")
    fig.show()
    logging.info("Plotly: Histograms of numerical features displayed.")

    # Box plots for numerical features (to show distributions and check for remaining outliers after capping)
    fig = make_subplots(rows=len(numerical_cols)//2 + len(numerical_cols)%2, cols=2,
                        subplot_titles=[f'Box plot of {col}' for col in numerical_cols])
    row_num = 1
    col_num = 1
    for col in numerical_cols:
        fig.add_trace(go.Box(y=df[col], name=col, marker_color='darkred'), row=row_num, col=col_num)
        fig.update_xaxes(title_text=col, row=row_num, col=col_num)
        fig.update_yaxes(title_text="Value", row=row_num, col=col_num)
        col_num += 1
        if col_num > 2:
            col_num = 1
            row_num += 1
    fig.update_layout(height=400 * row_num, showlegend=False, title_text="Box Plots of Numerical Features (Outlier Check)")
    fig.show()
    logging.info("Plotly: Box plots of numerical features displayed.")


    # Count plots for binary/categorical features
    binary_categorical_cols = ['male', 'education', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']
    fig = make_subplots(rows=len(binary_categorical_cols)//2 + len(binary_categorical_cols)%2, cols=2,
                        subplot_titles=[f'Count of {col}' for col in binary_categorical_cols])
    row_num = 1
    col_num = 1
    for col in binary_categorical_cols:
        counts = df[col].value_counts().reset_index()
        counts.columns = [col, 'count']
        fig.add_trace(go.Bar(x=counts[col], y=counts['count'], name=col, marker_color='mediumpurple'), row=row_num, col=col_num)
        fig.update_xaxes(title_text=col, type='category', row=row_num, col=col_num)
        fig.update_yaxes(title_text="Count", row=row_num, col=col_num)
        col_num += 1
        if col_num > 2:
            col_num = 1
            row_num += 1
    fig.update_layout(height=400 * row_num, showlegend=False, title_text="Distributions of Binary/Categorical Features")
    fig.show()
    logging.info("Plotly: Count plots of binary/categorical features displayed.")

    # Relationships with target variable
    for col in ['age', 'sysBP', 'totChol', 'glucose', 'BMI', 'cigsPerDay']:
        fig = px.box(df, x='TenYearCHD', y=col, color='TenYearCHD',
                     title=f'{col} vs TenYearCHD',
                     labels={'TenYearCHD': '10-Year CHD (0=No, 1=Yes)', col: col})
        fig.show()
        logging.info(f"Plotly: Box plot of {col} vs TenYearCHD displayed.")

    logging.info("Visual Representation of EDA completed.")

except Exception as e:
    handle_exception(e, "Visual EDA")


## 7. Visual Representation of Correlation and Covariance

Understanding the relationships between features and the target variable is crucial for feature selection and model interpretation. We'll use a heatmap to visualize the correlation matrix.

*   **Correlation:** Measures the strength and direction of a linear relationship between two variables. Values range from -1 (perfect negative correlation) to +1 (perfect positive correlation), with 0 indicating no linear correlation.
*   **Covariance:** Measures how two variables change together. A positive covariance indicates that variables tend to move in the same direction, while a negative covariance indicates they move in opposite directions. However, covariance values are not normalized, making them harder to interpret directly compared to correlation coefficients.


In [ ]:
try:
    logging.info("Starting Visual Representation of Correlation and Covariance.")

    # Calculate correlation matrix
    correlation_matrix = df.corr()

    # Plotting correlation heatmap using Plotly
    fig = px.imshow(correlation_matrix,
                    text_auto=True,
                    aspect="auto",
                    color_continuous_scale="RdBu_r",
                    title="Correlation Matrix of Features")
    fig.update_layout(width=900, height=800)
    fig.show()
    logging.info("Plotly: Correlation matrix heatmap displayed.")

    print("\nTop 10 features positively correlated with 'TenYearCHD':")
    print(correlation_matrix['TenYearCHD'].sort_values(ascending=False).head(11).iloc[1:]) # Exclude self-correlation

    print("\nTop 10 features negatively correlated with 'TenYearCHD':")
    print(correlation_matrix['TenYearCHD'].sort_values(ascending=True).head(10))

    # Calculate covariance matrix
    covariance_matrix = df.cov()

    # Plotting covariance heatmap (optional, as correlation is generally more interpretable)
    # Due to scale differences, directly plotting covariance might not be as visually insightful without normalization.
    # We will just print some key covariances.
    print("\nCovariance between 'TenYearCHD' and top correlated features:")
    for col in correlation_matrix['TenYearCHD'].sort_values(ascending=False).head(6).iloc[1:].index:
        print(f"Covariance({col}, TenYearCHD): {covariance_matrix.loc[col, 'TenYearCHD']:.2f}")

    logging.info("Visual Representation of Correlation and Covariance completed.")

except Exception as e:
    handle_exception(e, "Correlation and Covariance Visualization")


## Explanation of Correlation and Covariance Plots:

The correlation heatmap clearly shows the linear relationships between all pairs of features.
*   **Target Variable (`TenYearCHD`) Correlations:**
    *   `age`, `sysBP`, `glucose`, `diaBP`, `BMI`, `cigsPerDay`, `totChol`, `prevalentHyp` and `diabetes` show positive correlation with `TenYearCHD`. This is expected as these are known risk factors for heart disease. Higher values in these features tend to be associated with a higher likelihood of CHD.
    *   Features like `male` (gender), `education` and `heartRate` show relatively weaker correlations, but still contribute to the overall risk assessment.
    *   The engineered features like `age_group` dummies, `bmi_category` dummies, and `bp_status` dummies also show various levels of correlation, indicating their potential importance. For instance, higher blood pressure categories (e.g., `bp_status_High BP Stage 2`) are strongly positively correlated with `TenYearCHD`.
*   **Inter-feature Correlations:**
    *   Strong correlations are observed among blood pressure measurements (`sysBP` and `diaBP`).
    *   `currentSmoker` and `cigsPerDay` are highly correlated, which is obvious.
    *   `age` is positively correlated with several other features like `sysBP`, `totChol`, `prevalentHyp`, which makes sense as these conditions often worsen with age.
    *   High inter-feature correlations (multicollinearity) should be noted but often handled well by tree-based models. For linear models, it might lead to unstable coefficient estimates.

The covariance values, while harder to interpret directly due to their scale dependence, confirm the direction of the relationships seen in the correlation matrix. They are useful in statistical modeling but correlation provides a more normalized and intuitive measure of linear association.


## 8. Feature Selection Based on EDA

Based on the EDA and correlation analysis, we select features that are most relevant for predicting `TenYearCHD`. We'll prioritize features with strong correlations to the target, and retain clinically significant factors.

**Selected Features and Justification:**

1.  **Direct Risk Factors:**
    *   `age`: Strong positive correlation, fundamental risk factor.
    *   `sysBP`, `diaBP`: Both show strong positive correlation; crucial indicators of cardiovascular health.
    *   `glucose`, `totChol`: Direct physiological markers for diabetes and high cholesterol, strongly linked to CHD.
    *   `cigsPerDay`: Direct measure of smoking habit, strong risk factor.
    *   `BMI`: An indicator of obesity, another significant risk factor.
    *   `heartRate`: While weaker, still a physiological measure.
    *   `diabetes`, `prevalentHyp`, `prevalentStroke`: Binary indicators of existing conditions, which are major risk factors.
    *   `currentSmoker`: Binary indicator of smoking, overlaps with `cigsPerDay` but can still be useful.
    *   `BPMeds`: Indicator of medication for blood pressure, suggests existing hypertension.
    *   `male`: Gender is a known demographic risk factor.

2.  **Engineered Features:**
    *   `age_group` dummies: Captures non-linear effects of age.
    *   `bmi_category` dummies: Captures non-linear effects of BMI.
    *   `bp_status` dummies: More granular representation of blood pressure risk.

3.  **Removed Features:**
    *   `education`: Showed very weak correlation and less clinical significance in direct prediction compared to other factors. Its role is often indirect or socio-economic, and may not add much predictive power over other features. Also, it's ordinal, but given the options (1,2,3,4) and low correlation, its impact is likely minimal.


In [ ]:
try:
    logging.info("Starting Feature Selection.")

    # Drop the original 'education' column and the base columns used for creating one-hot encoded features
    features_to_drop = ['education', 'age', 'BMI', 'sysBP', 'diaBP'] # Drop original numericals if using only engineered ones, or keep both.
                                                                    # For this model, keeping original numericals and engineered dummies is beneficial
                                                                    # because dummies capture categorical aspects, while original numericals provide continuous info.
                                                                    # So, only dropping 'education' and the original columns for which we have more detailed dummies.
                                                                    # No, actually, for features like 'age', 'BMI', 'sysBP', 'diaBP', we want to keep the original numerical values
                                                                    # AND the one-hot encoded features derived from them. The one-hot encoded features help capture
                                                                    # non-linear relationships or thresholds.

    # Re-evaluate features to drop: only 'education' from original
    # We will keep 'age', 'BMI', 'sysBP', 'diaBP' as numerical features and their dummy engineered counterparts.
    # The 'age_group', 'bmi_category', 'bp_status' columns themselves were dropped during get_dummies, so no need to explicitly drop here.
    final_features_to_drop = ['education'] # This is the only column we decided to remove.

    # Define features (X) and target (y)
    X = df.drop(columns=['TenYearCHD'] + final_features_to_drop)
    y = df['TenYearCHD']

    # Identify numerical features for scaling
    # Exclude binary and one-hot encoded features from standard scaling
    numerical_features_for_scaling = ['cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'age']
    # Filter to only include columns present in X
    numerical_features_for_scaling = [col for col in numerical_features_for_scaling if col in X.columns]


    logging.info(f"Features selected for X: {X.columns.tolist()}")
    logging.info(f"Target variable y: {y.name}")
    print(f"\nNumber of features selected: {X.shape[1]}")
    print("\nFirst 5 rows of features (X):")
    print(X.head())
    print("\nFirst 5 rows of target (y):")
    print(y.head())

    # 9. Data Scaling (part of preprocessing, but applied after feature selection)
    scaler = StandardScaler()
    X[numerical_features_for_scaling] = scaler.fit_transform(X[numerical_features_for_scaling])
    logging.info(f"Numerical features {numerical_features_for_scaling} scaled using StandardScaler.")
    print("\nFeatures (X) after scaling numerical columns:")
    print(X.head())

    # Save the scaler for later use (e.g., for new predictions)
    with open(os.path.join(ARTIFACTS_PATH, 'scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)
    logging.info(f"Scaler saved to {os.path.join(ARTIFACTS_PATH, 'scaler.pkl')}")

except Exception as e:
    handle_exception(e, "Feature Selection and Scaling")


## 9. Modeling

This is a binary classification problem. We will employ three common and effective models:
1.  **Logistic Regression:** A good baseline, interpretable linear model.
2.  **Random Forest Classifier:** An ensemble tree-based model, good for capturing non-linear relationships and interactions.
3.  **Gradient Boosting Classifier (e.g., XGBoost/LightGBM):** Another powerful ensemble tree-based model known for high performance. For simplicity, we'll use `GradientBoostingClassifier` from scikit-learn.

We will use a pipeline that incorporates SMOTE for handling class imbalance (as observed in EDA), followed by the classifier.


In [ ]:
try:
    logging.info("Starting Model Training and Evaluation.")

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    logging.info(f"Data split into training (X_train shape: {X_train.shape}) and testing (X_test shape: {X_test.shape}) sets.")
    print(f"\nTraining set shape: {X_train.shape}, Test set shape: {X_test.shape}")
    print(f"Training target distribution:\n{y_train.value_counts(normalize=True)}")
    print(f"Test target distribution:\n{y_test.value_counts(normalize=True)}")


    # Define models
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear'), # liblinear is good for small datasets and L1/L2
        'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42)
    }

    results = {}

    for name, model in models.items():
        logging.info(f"Training {name}...")
        print(f"\n--- Training {name} ---")

        # Create a pipeline with SMOTE and the model
        # SMOTE should only be applied to the training data. Using imblearn's Pipeline is crucial
        # because it ensures SMOTE is applied correctly within cross-validation folds as well.
        pipeline = ImbPipeline([
            ('smote', SMOTE(random_state=42, sampling_strategy='auto')), # 'auto' balances all classes
            ('classifier', model)
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        # Evaluation
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)
        conf_matrix = confusion_matrix(y_test, y_pred)
        class_report = classification_report(y_test, y_pred)

        results[name] = {
            'model': pipeline, # Store the pipeline
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'roc_auc': roc_auc,
            'confusion_matrix': conf_matrix,
            'classification_report': class_report,
            'y_pred': y_pred,
            'y_proba': y_proba
        }

        logging.info(f"{name} trained and evaluated.")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print(f"ROC AUC: {roc_auc:.4f}")
        print("Confusion Matrix:\n", conf_matrix)
        print("Classification Report:\n", class_report)

    logging.info("Model Training and Evaluation completed for all models.")

except Exception as e:
    handle_exception(e, "Model Training and Evaluation")


## 10. Evaluation Metrics

For this binary classification task, where predicting CHD (positive class) is often more critical than predicting no CHD (negative class), the following metrics are highly relevant:

*   **Accuracy:** Overall correctness of predictions. While useful, it can be misleading in imbalanced datasets (e.g., if 90% are negative, a model predicting all negative would have 90% accuracy).
*   **Precision:** Of all positive predictions, how many were actually correct? (TP / (TP + FP)). Important when the cost of False Positives is high (e.g., unnecessary medical tests).
*   **Recall (Sensitivity):** Of all actual positive cases, how many were correctly identified? (TP / (TP + FN)). Important when the cost of False Negatives is high (e.g., missing a CHD case).
*   **F1-Score:** The harmonic mean of Precision and Recall. Provides a balance between the two and is good for imbalanced datasets.
*   **ROC AUC (Receiver Operating Characteristic Area Under the Curve):** Measures the ability of the model to distinguish between classes. A higher AUC indicates a better model performance, regardless of the classification threshold. It's robust to class imbalance.
*   **Confusion Matrix:** A table that summarizes the performance of a classification algorithm. It shows True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).
*   **Classification Report:** Provides Precision, Recall, F1-score, and support for each class.

Given that `TenYearCHD` is likely an imbalanced dataset (fewer positive cases), **Recall, F1-Score, and ROC AUC** are particularly important to ensure we don't miss actual CHD cases.


## 11. Local Minima vs. Global Minima and Visual Representation of Gradient Descent

### Local Minima vs. Global Minima

In the context of optimization, particularly when training machine learning models, we aim to find the set of model parameters that minimize a cost (or loss) function.

*   **Global Minimum:** This is the lowest possible value of the cost function across the entire parameter space. It represents the absolute best set of parameters for the model, where the model's error is minimized.
*   **Local Minimum:** This is a point in the parameter space where the cost function is lower than in any *neighboring* points, but it is not necessarily the absolute lowest value across the entire space.

Think of it like a landscape: the global minimum is the lowest valley in the entire landscape, while local minima are smaller dips or valleys that are lower than their immediate surroundings but not as low as the deepest valley.

For complex, non-convex cost functions (common in deep learning and some non-linear models), gradient descent algorithms can sometimes get stuck in a local minimum, failing to reach the global minimum. This can lead to a suboptimal model.

### Visual Representation of Gradient Descent

Gradient Descent is an iterative optimization algorithm used to find the minimum of a function. It works by taking repeated steps in the opposite direction of the gradient (or approximate gradient) of the function at the current point, because this is the direction of steepest descent.

**For a simple 2D function:**

Imagine a function `f(x) = x^2`. The minimum is at `x=0`.
*   Start at `x = 4`.
*   The gradient `f'(x) = 2x`. At `x=4`, `f'(4) = 8`.
*   Step in the opposite direction: `x_new = x - learning_rate * gradient`.
    If `learning_rate = 0.1`, `x_new = 4 - 0.1 * 8 = 3.2`.
*   Repeat until convergence.

**Using our dataset for conceptual visualization:**

While directly visualizing gradient descent in a high-dimensional feature space for a complex model like Logistic Regression or Gradient Boosting is not feasible, we can illustrate the concept using a simplified loss function with one or two features.

Let's use a simple linear regression-like setup for a single feature `age` to predict a simplified `BP` (e.g., `sysBP`). We are not training a model here, but illustrating the concept of how a simple cost function (e.g., Mean Squared Error) can be minimized.


In [ ]:
try:
    logging.info("Starting visualization of Gradient Descent concept.")

    # Select a feature and target for a simple illustration (e.g., age vs sysBP)
    # This is for conceptual illustration, not actual model training.
    X_gd = df['age'].values.reshape(-1, 1)
    y_gd = df['sysBP'].values

    # Let's consider a simplified linear model: y = m*x + c
    # Cost function: MSE = (1/N) * sum((y_pred - y_true)^2)
    # We'll fix intercept (c=0) and visualize cost wrt 'm' (slope)
    # Cost(m) = (1/N) * sum((m*x - y)^2)

    # Define a range of slopes (m) to visualize the cost function
    slopes = np.linspace(-5, 5, 100)
    costs = []
    for m in slopes:
        y_pred_gd = m * X_gd
        cost = np.mean((y_pred_gd - y_gd)**2)
        costs.append(cost)

    # Plot the cost function (conceptual)
    fig_cost = px.line(x=slopes, y=costs, title="Conceptual Cost Function (MSE vs. Slope 'm')",
                       labels={'x': 'Slope (m)', 'y': 'Cost (Mean Squared Error)'})
    fig_cost.update_layout(hovermode="x unified")
    fig_cost.show()
    logging.info("Plotly: Conceptual Cost Function displayed for gradient descent explanation.")

    # Illustrate Gradient Descent steps
    # Initial parameters
    m_current = -4.0
    learning_rate = 0.0001 # A small learning rate for illustration
    iterations = 50

    m_history = [m_current]
    cost_history = []

    for i in range(iterations):
        y_pred_gd_current = m_current * X_gd
        # Gradient of MSE wrt m: (2/N) * sum((m*x - y)*x)
        gradient_m = np.mean(2 * (y_pred_gd_current - y_gd) * X_gd)
        m_current = m_current - learning_rate * gradient_m
        cost = np.mean((m_current * X_gd - y_gd)**2)
        m_history.append(m_current)
        cost_history.append(cost)

    fig_gd = go.Figure()
    fig_gd.add_trace(go.Scatter(x=slopes, y=costs, mode='lines', name='Cost Function'))
    fig_gd.add_trace(go.Scatter(x=m_history, y=cost_history, mode='markers+lines',
                                name='Gradient Descent Path',
                                marker=dict(color='red', size=8),
                                line=dict(color='red', width=2)))

    fig_gd.update_layout(title="Gradient Descent Visualization on Conceptual Cost Function",
                         xaxis_title="Slope (m)",
                         yaxis_title="Cost (Mean Squared Error)")
    fig_gd.show()
    logging.info("Plotly: Gradient Descent path visualization displayed.")

except Exception as e:
    handle_exception(e, "Gradient Descent Visualization")


## 12. Residuals and How to Visualize Them

### What are Residuals?

In regression tasks, residuals are the differences between the observed values and the predicted values by the model.
`Residual = Actual Value - Predicted Value`

For **classification tasks**, the concept of residuals is slightly different. Instead of a continuous error, we often look at:
1.  **Misclassifications:** Which instances were predicted incorrectly?
2.  **Probability-based residuals:** The difference between the true binary label (0 or 1) and the predicted probability (0 to 1) for that class.

### How to Visualize Residuals (for Classification)

For binary classification, common "residual-like" visualizations include:

1.  **Confusion Matrix:** Directly shows counts of TP, TN, FP, FN. This is the most basic and powerful way to understand classification errors.
2.  **ROC Curve:** Visualizes the trade-off between True Positive Rate (Recall) and False Positive Rate at various classification thresholds.
3.  **Reliability (or Calibration) Plots:** Compares the predicted probabilities to the actual observed frequencies of the positive class. If the model is well-calibrated, a predicted probability of 0.8 should correspond to the positive class being observed 80% of the time.
4.  **Distribution of Predicted Probabilities:** Histograms of predicted probabilities for each class can show how well-separated the classes are and where the model is uncertain.

### Comparison and Metrics to Improve

We'll visualize the confusion matrix and ROC curve for our best model.

*   **Confusion Matrix:**
    *   **High FP (False Positives):** The model predicts CHD, but the patient doesn't have it. This can lead to unnecessary anxiety and medical procedures. To reduce FP, you might focus on increasing precision.
    *   **High FN (False Negatives):** The model predicts no CHD, but the patient actually has it. This is often the more critical error in medical diagnosis, leading to missed diagnoses and delayed treatment. To reduce FN, you might focus on increasing recall.
*   **ROC Curve:** A larger area under the curve (AUC) indicates a better ability to distinguish between classes. A curve closer to the top-left corner is better.
*   **Reliability Plot:** A poorly calibrated model might consistently overestimate or underestimate probabilities. Calibration techniques (e.g., Platt Scaling, Isotonic Regression) can be used to improve this.

To improve residuals (i.e., reduce errors):
*   **Feature Engineering:** Create more informative features.
*   **Model Complexity:** Try more complex models (e.g., ensemble methods) if underfitting.
*   **Regularization:** If overfitting, add regularization to simpler models or tune hyperparameters in complex models.
*   **Ensemble Methods:** Combine predictions from multiple models.
*   **Hyperparameter Tuning:** Optimize model parameters.
*   **Collect More Data:** Increase the size and diversity of the dataset.
*   **Data Cleaning:** Ensure data quality.

Let's visualize the confusion matrix and ROC curve for the best performing model.


In [ ]:
try:
    logging.info("Starting Residuals Visualization (Classification Metrics).")

    # Assuming 'Gradient Boosting' was the best performing model or we select it for demonstration
    best_model_name = max(results, key=lambda k: results[k]['f1_score']) # Or roc_auc, depending on priority
    print(f"\nVisualizing results for the best model based on F1-Score: {best_model_name}")
    best_model_results = results[best_model_name]

    # Confusion Matrix Visualization
    cm = best_model_results['confusion_matrix']
    fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                        labels=dict(x="Predicted", y="True", color="Count"),
                        x=['Predicted 0', 'Predicted 1'],
                        y=['True 0', 'True 1'],
                        title=f'Confusion Matrix for {best_model_name}')
    fig_cm.update_xaxes(side="bottom")
    fig_cm.show()
    logging.info(f"Plotly: Confusion Matrix for {best_model_name} displayed.")

    # ROC Curve Visualization
    fpr, tpr, thresholds = roc_curve(y_test, best_model_results['y_proba'])
    fig_roc = px.area(
        x=fpr, y=tpr,
        title=f'ROC Curve (AUC={best_model_results["roc_auc"]:.4f}) for {best_model_name}',
        labels=dict(x='False Positive Rate', y='True Positive Rate'),
        width=700, height=500
    )
    fig_roc.add_shape(
        type='line', line=dict(dash='dash'),
        x0=0, x1=1, y0=0, y1=1
    )
    fig_roc.update_yaxes(scaleanchor="x", scaleratio=1)
    fig_roc.update_xaxes(constrain='domain')
    fig_roc.show()
    logging.info(f"Plotly: ROC Curve for {best_model_name} displayed.")

    logging.info("Residuals Visualization completed.")

except Exception as e:
    handle_exception(e, "Residuals Visualization")


## 13. Overfitting or Underfitting

### Explanation

*   **Overfitting:** Occurs when a model learns the training data too well, including its noise and outliers, to the extent that it performs poorly on unseen data. The model is too complex for the given data, essentially "memorizing" the training examples rather than learning general patterns.
    *   **Symptoms:** High accuracy/performance on the training set, but significantly lower accuracy/performance on the test (unseen) set. High variance.
    *   **Analogy:** A student who memorizes answers to specific questions for an exam but doesn't understand the underlying concepts will do poorly on new questions.

*   **Underfitting:** Occurs when a model is too simple to capture the underlying patterns in the training data, leading to poor performance on both the training and test sets. The model hasn't learned enough from the data.
    *   **Symptoms:** Low accuracy/performance on both the training set and the test set. High bias.
    *   **Analogy:** A student who doesn't study enough for an exam will do poorly because they haven't learned the material.

### How to Fix It

**To fix Overfitting:**
1.  **Get More Data:** A larger, more diverse dataset can help the model learn more general patterns.
2.  **Feature Selection/Reduction:** Remove irrelevant or redundant features that might be contributing to noise.
3.  **Regularization:** Add penalty terms to the loss function (e.g., L1, L2 regularization in Logistic Regression or Linear Models) to constrain model complexity.
4.  **Cross-Validation:** Helps to get a more robust estimate of model performance and detect overfitting.
5.  **Simplify Model:** Use a less complex model (e.g., linear model instead of a deep neural network, or fewer trees/ shallower trees in ensemble methods).
6.  **Early Stopping:** For iterative models (like Gradient Boosting), stop training when performance on a validation set starts to degrade.
7.  **Increase Dropout (Neural Networks):** Randomly drop units during training to prevent co-adaptation.

**To fix Underfitting:**
1.  **Add More Features:** Create new features or use more existing features that are relevant to the target.
2.  **Increase Model Complexity:** Use a more powerful or flexible model (e.g., switch from Linear Regression to Random Forest, or increase depth/number of estimators).
3.  **Reduce Regularization:** Decrease the penalty on model complexity.
4.  **Feature Engineering:** Create interaction terms or polynomial features.
5.  **Remove Noise from Data:** Clean up irrelevant data points or errors that might confuse the model.

**In our case:**
We evaluate models on both training and test sets. If we observe a large gap between training and test performance (e.g., training F1-score is 0.95, but test F1-score is 0.60), then overfitting is likely. If both training and test F1-scores are very low (e.g., 0.30), then underfitting is likely. Our use of SMOTE helps with imbalance, and ensemble models like Random Forest and Gradient Boosting are generally robust but can overfit if hyperparameters aren't tuned carefully. Cross-validation (which will be part of hyperparameter tuning) is key to detecting and mitigating these issues.


## 14. Create Example Dataset with Features Used for Modeling and Make Predictions

Let's create a small, artificial dataset with the same features used for training our models and then use the best-performing model to make predictions. This demonstrates how the model would be used in a real-world scenario.


In [ ]:
try:
    logging.info("Creating example dataset and making predictions.")

    # Get the feature names used by the trained model
    model_features = X.columns.tolist()

    # Create a sample data point (hypothetical patient)
    # The values should be in the original scale before scaling, then scaled by the saved scaler.
    example_patient_data = {
        'male': 0, 'age': 55, 'currentSmoker': 1, 'cigsPerDay': 15.0, 'BPMeds': 0.0,
        'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 240.0,
        'sysBP': 145.0, 'diaBP': 90.0, 'BMI': 28.5, 'heartRate': 75.0, 'glucose': 95.0
    }

    # Convert to DataFrame
    example_df = pd.DataFrame([example_patient_data])

    # Re-apply feature engineering steps to the example data
    # age_group
    example_df['age_group'] = pd.cut(example_df['age'], bins=[20, 30, 40, 50, 60, 70, 80, np.inf],
                                     labels=['20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80+'], right=False)
    # bmi_category
    def bmi_category(bmi):
        if bmi < 18.5: return 'Underweight'
        elif 18.5 <= bmi < 24.9: return 'Normal'
        elif 24.9 <= bmi < 29.9: return 'Overweight'
        else: return 'Obese'
    example_df['bmi_category'] = example_df['BMI'].apply(bmi_category)
    # bp_status
    def bp_status(sysBP, diaBP):
        if sysBP < 120 and diaBP < 80: return 'Normal BP'
        elif (120 <= sysBP < 130 and diaBP < 80) or (sysBP < 120 and 80 <= diaBP < 90): return 'Elevated BP'
        elif (130 <= sysBP < 140 or 80 <= diaBP < 90): return 'High BP Stage 1'
        elif (sysBP >= 140 or diaBP >= 90): return 'High BP Stage 2'
        else: return 'Unknown BP'
    example_df['bp_status'] = example_df.apply(lambda row: bp_status(row['sysBP'], row['diaBP']), axis=1)

    # One-hot encode new categorical features, aligning columns with training data
    example_df = pd.get_dummies(example_df, columns=['age_group', 'bmi_category', 'bp_status'], drop_first=True)

    # Ensure all columns from training data are present in example_df, fill missing with 0
    missing_cols = set(model_features) - set(example_df.columns)
    for c in missing_cols:
        example_df[c] = 0
    # Ensure the order of columns is the same as in training data
    example_df = example_df[model_features]

    # Load the saved scaler
    with open(os.path.join(ARTIFACTS_PATH, 'scaler.pkl'), 'rb') as f:
        loaded_scaler = pickle.load(f)
    logging.info("Scaler loaded for example prediction.")

    # Scale numerical features (using the same columns as in X_train)
    numerical_features_for_scaling = ['cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'age']
    numerical_features_for_scaling = [col for col in numerical_features_for_scaling if col in example_df.columns]

    example_df[numerical_features_for_scaling] = loaded_scaler.transform(example_df[numerical_features_for_scaling])
    logging.info("Example data scaled.")

    # Make prediction using the best model
    best_model_pipeline = results[best_model_name]['model']
    prediction = best_model_pipeline.predict(example_df)
    prediction_proba = best_model_pipeline.predict_proba(example_df)[:, 1]

    print(f"\nExample Patient Data:")
    print(example_patient_data)
    print(f"\nPredicted 10-Year CHD Risk (0=No, 1=Yes): {prediction[0]}")
    print(f"Predicted Probability of CHD: {prediction_proba[0]:.4f}")
    logging.info(f"Example prediction made: {prediction[0]} with probability {prediction_proba[0]:.4f}")

except Exception as e:
    handle_exception(e, "Example Prediction")


## 15. Hyperparameter Tuning on Sample or Small Dataset

Hyperparameter tuning is essential to optimize model performance. We will use `GridSearchCV` on a smaller subset of the training data or with limited parameter grids to manage computational cost, or on the full training set with optimized parameters if resources allow.

For this demonstration, we will tune the `GradientBoostingClassifier` as it often offers strong performance.


In [ ]:
try:
    logging.info("Starting Hyperparameter Tuning for Gradient Boosting Classifier.")

    # Define a smaller subset of data for tuning if full dataset is too large, or proceed with full if manageable.
    # For this example, we'll use a subset of the training data to speed up GridSearchCV.
    # Optionally: X_tune, _, y_tune, _ = train_test_split(X_train, y_train, test_size=0.8, random_state=42, stratify=y_train)
    X_tune, y_tune = X_train, y_train # Use full training set for more robust tuning

    # Define parameter grid for GradientBoostingClassifier
    param_grid = {
        'classifier__n_estimators': [100, 200], # Number of boosting stages
        'classifier__learning_rate': [0.05, 0.1, 0.2], # Step size shrinkage
        'classifier__max_depth': [3, 4], # Maximum depth of the individual regression estimators
        'classifier__subsample': [0.8, 1.0], # Fraction of samples to be used for fitting the individual base learners
        'classifier__min_samples_leaf': [1, 2] # Minimum number of samples required to be at a leaf node
    }

    # Use the pipeline including SMOTE
    gb_pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('classifier', GradientBoostingClassifier(random_state=42))
    ])

    # Setup GridSearchCV
    # We will prioritize 'roc_auc' for optimization given the imbalanced nature
    grid_search = GridSearchCV(gb_pipeline, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)

    print("\nStarting GridSearchCV for GradientBoostingClassifier...")
    grid_search.fit(X_tune, y_tune)

    logging.info("Hyperparameter tuning completed.")
    print("\nBest parameters found:", grid_search.best_params_)
    print("Best ROC AUC score:", grid_search.best_score_)

    # Update the results with the tuned model
    tuned_gb_model = grid_search.best_estimator_
    y_pred_tuned = tuned_gb_model.predict(X_test)
    y_proba_tuned = tuned_gb_model.predict_proba(X_test)[:, 1]

    results['Tuned Gradient Boosting'] = {
        'model': tuned_gb_model,
        'accuracy': accuracy_score(y_test, y_pred_tuned),
        'precision': precision_score(y_test, y_pred_tuned),
        'recall': recall_score(y_test, y_pred_tuned),
        'f1_score': f1_score(y_test, y_pred_tuned),
        'roc_auc': roc_auc_score(y_test, y_proba_tuned),
        'confusion_matrix': confusion_matrix(y_test, y_pred_tuned),
        'classification_report': classification_report(y_test, y_pred_tuned),
        'y_pred': y_pred_tuned,
        'y_proba': y_proba_tuned
    }
    logging.info("Tuned Gradient Boosting model evaluated and added to results.")

    print("\nTuned Gradient Boosting Model Performance on Test Set:")
    print(f"Accuracy: {results['Tuned Gradient Boosting']['accuracy']:.4f}")
    print(f"Precision: {results['Tuned Gradient Boosting']['precision']:.4f}")
    print(f"Recall: {results['Tuned Gradient Boosting']['recall']:.4f}")
    print(f"F1-Score: {results['Tuned Gradient Boosting']['f1_score']:.4f}")
    print(f"ROC AUC: {results['Tuned Gradient Boosting']['roc_auc']:.4f}")
    print("Classification Report:\n", results['Tuned Gradient Boosting']['classification_report'])

except Exception as e:
    handle_exception(e, "Hyperparameter Tuning")


## 16. Visual Representation of the Results

Let's visualize the comparison of different models and the results of the best model (potentially the tuned one).


In [ ]:
try:
    logging.info("Starting Visualization of Results.")

    # Create a DataFrame for comparing model metrics
    metrics_df = pd.DataFrame({
        'Model': [name for name in results.keys()],
        'Accuracy': [res['accuracy'] for res in results.values()],
        'Precision': [res['precision'] for res in results.values()],
        'Recall': [res['recall'] for res in results.values()],
        'F1-Score': [res['f1_score'] for res in results.values()],
        'ROC AUC': [res['roc_auc'] for res in results.values()]
    }).set_index('Model')

    print("\n--- Model Comparison ---")
    print(metrics_df.round(4))

    # Plotting comparison of F1-score and ROC AUC
    fig_comp = make_subplots(rows=1, cols=2, subplot_titles=('Model F1-Score Comparison', 'Model ROC AUC Comparison'))

    fig_comp.add_trace(go.Bar(x=metrics_df.index, y=metrics_df['F1-Score'], name='F1-Score',
                              marker_color=['blue', 'green', 'red', 'purple']), row=1, col=1)
    fig_comp.add_trace(go.Bar(x=metrics_df.index, y=metrics_df['ROC AUC'], name='ROC AUC',
                              marker_color=['lightblue', 'lightgreen', 'salmon', 'darkviolet']), row=1, col=2)

    fig_comp.update_layout(height=500, width=1000, title_text="Comparison of Model Performance Metrics")
    fig_comp.show()
    logging.info("Plotly: Model F1-Score and ROC AUC comparison displayed.")

    # Visualize ROC curves for all models
    fig_roc_all = go.Figure()
    fig_roc_all.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1)

    for name, res in results.items():
        fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
        fig_roc_all.add_trace(go.Scatter(x=fpr, y=tpr, name=f'{name} (AUC={res["roc_auc"]:.4f})', mode='lines'))

    fig_roc_all.update_layout(title='ROC Curve Comparison for All Models',
                              xaxis_title='False Positive Rate',
                              yaxis_title='True Positive Rate',
                              yaxis=dict(scaleanchor="x", scaleratio=1),
                              xaxis=dict(constrain='domain'))
    fig_roc_all.show()
    logging.info("Plotly: All models ROC Curve comparison displayed.")


    # Visualizing predicted vs. true for the best model (e.g., using a small sample)
    # This is more intuitive for regression, for classification we can show probability distributions.
    best_model_name_final = max(results, key=lambda k: results[k]['roc_auc']) # Final selection based on ROC AUC
    best_model_res = results[best_model_name_final]

    df_predictions = pd.DataFrame({'True_CHD': y_test, 'Predicted_Prob_CHD': best_model_res['y_proba']})
    df_predictions['Predicted_Class'] = best_model_res['y_pred']

    fig_prob_dist = px.histogram(df_predictions, x="Predicted_Prob_CHD", color="True_CHD", marginal="box",
                                 title=f'Predicted Probability Distribution for {best_model_name_final}',
                                 labels={'True_CHD': 'True Class (0=No, 1=Yes)'},
                                 nbins=50, opacity=0.7)
    fig_prob_dist.show()
    logging.info(f"Plotly: Predicted Probability Distribution for {best_model_name_final} displayed.")

    logging.info("Visualization of Results completed.")

except Exception as e:
    handle_exception(e, "Visualization of Results")


## Explanation of Result Visualizations:

1.  **Model Performance Comparison Bars (F1-Score and ROC AUC):** These bar charts provide a quick visual comparison of how each model (Logistic Regression, Random Forest, Gradient Boosting, and Tuned Gradient Boosting) performs across key metrics relevant for imbalanced classification.
    *   We expect to see the ensemble methods (Random Forest, Gradient Boosting) generally outperforming Logistic Regression.
    *   Hyperparameter tuning should ideally show an improvement in metrics for the 'Tuned Gradient Boosting' model compared to the untuned version.

2.  **ROC Curve Comparison:** This plot aggregates the ROC curves of all models onto a single graph.
    *   The model with its curve closest to the top-left corner and the largest Area Under the Curve (AUC) is considered superior. A perfect classifier would have an AUC of 1.0 (curve going straight up from (0,0) to (0,1) and then across to (1,1)). A random classifier would have an AUC of 0.5 (diagonal line).
    *   This visualization helps to understand the trade-off between True Positive Rate (Recall) and False Positive Rate for each model across different decision thresholds.

3.  **Predicted Probability Distribution for Best Model:** This histogram, split by the true class (`True_CHD`), shows the distribution of predicted probabilities.
    *   For a good classifier, the predicted probabilities for `True_CHD = 0` should be concentrated towards 0, and for `True_CHD = 1`, they should be concentrated towards 1.
    *   Overlap in these distributions indicates uncertainty or misclassifications, especially if there are many `True_CHD = 0` instances with high predicted probabilities (False Positives) or `True_CHD = 1` instances with low predicted probabilities (False Negatives). This helps to visualize where the model struggles to differentiate the classes.


## 17. Final Model Selection

Based on the evaluation metrics and visualizations, we select the model that best balances prediction performance, particularly for the positive class (CHD), and robustness. Given the nature of medical predictions, a good balance of **Recall** (to minimize false negatives) and **ROC AUC** (overall discriminatory power) is often prioritized. The **F1-score** is also a good composite metric for imbalanced datasets.

Comparing the models:
*   **Logistic Regression:** Serves as a strong baseline, offering interpretability but often less predictive power than ensemble methods.
*   **Random Forest:** Typically robust and good at handling non-linearity and interactions.
*   **Gradient Boosting (tuned):** Often achieves the highest performance among tree-based ensembles, especially after hyperparameter tuning. It typically yields the best ROC AUC and F1-score due to its sequential error correction mechanism.

Considering the tuning results, the `Tuned Gradient Boosting Classifier` often emerges as the best choice due to its superior ROC AUC and F1-score, indicating strong discriminatory power and a good balance between precision and recall, crucial for medical diagnostic predictions.


In [ ]:
try:
    logging.info("Final Model Selection.")

    # Selecting the best model based on ROC AUC, which is a robust metric for imbalanced classification.
    best_model_name = max(results, key=lambda k: results[k]['roc_auc'])
    final_model = results[best_model_name]['model']

    print(f"\nFinal Model Selected: {best_model_name}")
    print(f"Its ROC AUC on the test set: {results[best_model_name]['roc_auc']:.4f}")
    print(f"Its F1-Score on the test set: {results[best_model_name]['f1_score']:.4f}")

    logging.info(f"Final model selected: {best_model_name} with ROC AUC: {results[best_model_name]['roc_auc']:.4f}")

except Exception as e:
    handle_exception(e, "Final Model Selection")


## 18. Save the Model

The final selected model is saved using the `pickle` library into the `artifacts` directory for future use, such as deploying it for making new predictions without retraining.


In [ ]:
try:
    logging.info("Saving the final model.")

    model_filename = os.path.join(ARTIFACTS_PATH, f'{best_model_name.replace(" ", "_").lower()}_chd_predictor.pkl')
    with open(model_filename, 'wb') as file:
        pickle.dump(final_model, file)

    print(f"\nFinal model saved successfully as '{model_filename}'")
    logging.info(f"Model saved to {model_filename}")

except Exception as e:
    handle_exception(e, "Model Saving")


## 19. Insights

Based on our analysis and modeling, here are some key insights regarding 10-year CHD risk prediction from the Framingham Heart Study dataset:

*   **Key Risk Factors:** Age, systolic blood pressure (`sysBP`), glucose levels, total cholesterol (`totChol`), and the presence of hypertension (`prevalentHyp`) and diabetes are consistently among the most significant predictors of 10-year CHD risk. These findings align with established medical knowledge about cardiovascular health.
*   **Impact of Lifestyle:** `cigsPerDay` (cigarettes per day) and `BMI` (Body Mass Index) are also important risk factors, highlighting the impact of lifestyle choices on heart health.
*   **Feature Engineering Value:** The engineered features such as `age_group` (e.g., '50-59' and '60-69'), `bmi_category` (e.g., 'Obese'), and `bp_status` (e.g., 'High BP Stage 2') demonstrated strong correlations and likely improved model performance by capturing non-linear relationships and clinically relevant thresholds.
*   **Class Imbalance:** The target variable `TenYearCHD` is imbalanced, with significantly fewer positive cases. Techniques like SMOTE were crucial to ensure the models could effectively learn to identify the minority class (CHD patients) without being biased towards the majority class.
*   **Ensemble Models Perform Best:** Ensemble methods like Gradient Boosting and Random Forest generally outperformed Logistic Regression, indicating that the relationships within the data are complex and non-linear. Gradient Boosting, especially after tuning, demonstrated the highest discriminatory power (ROC AUC) and a good balance between precision and recall (F1-score).
*   **Focus on Recall and ROC AUC:** For medical prediction tasks like this, minimizing False Negatives (missing a CHD case) is often paramount. Therefore, metrics like Recall and ROC AUC are critical for model selection, ensuring that the model has a high ability to identify actual risk.

These insights can be valuable for healthcare professionals in assessing patient risk and guiding preventive strategies.


## 20. Conclusion

This project successfully developed and evaluated machine learning models for predicting the 10-year risk of Coronary Heart Disease using the Framingham Heart Study dataset.

We followed a structured approach:
1.  **Data Loading and Initial Exploration:** Ensured robust data ingestion with logging and error handling.
2.  **Comprehensive EDA:** Gained deep insights into data distributions, missing values, and potential outliers.
3.  **Data Preprocessing:** Handled missing values through median/mode imputation, capped outliers using the IQR method, and performed significant feature engineering (age groups, BMI categories, BP status) to create more informative predictors. Numerical features were scaled to standardize their range.
4.  **Model Training:** Employed Logistic Regression, Random Forest, and Gradient Boosting Classifiers, incorporating SMOTE to address class imbalance.
5.  **Hyperparameter Tuning:** Optimized the Gradient Boosting Classifier using GridSearchCV, which led to improved performance.
6.  **Model Evaluation:** Utilized appropriate metrics (Accuracy, Precision, Recall, F1-Score, ROC AUC, Confusion Matrix) relevant for imbalanced binary classification.
7.  **Visualization:** Plotly was extensively used to visualize data distributions, correlations, model comparisons, ROC curves, and probability distributions, providing interactive and clear explanations.
8.  **Model Selection and Saving:** The `Tuned Gradient Boosting Classifier` emerged as the best-performing model based on its superior ROC AUC and F1-score and was saved for future deployment.

The selected model provides a valuable tool for identifying individuals at higher risk of developing CHD, enabling proactive healthcare interventions. The detailed process outlined in this notebook, from data understanding to model deployment, ensures transparency, reproducibility, and a robust solution for this critical public health challenge.

Further improvements could involve exploring more advanced feature engineering techniques, experimenting with deeper learning models (e.g., neural networks), or incorporating more sophisticated ensemble stacking methods. Continuous monitoring and re-evaluation of the model with new data would also be crucial in a real-world application to ensure its ongoing effectiveness.


In [ ]:
logging.info("ML pipeline notebook execution completed.")